# Jev quickstart

The smallest useful call to TypeSafe AI's Jev. You give it some text and a question. It returns an answer **plus a probability for every possible answer**.

Setup: put your key in `.env` at the repo root (`TYPESAFE_API_KEY=...`).

In [4]:
from pathlib import Path
import os

# Read TYPESAFE_API_KEY from the repo's .env (no extra packages needed)
for env in (Path(".env"), Path("../.env")):
    if env.exists():
        for line in env.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
print("key loaded" if API_KEY else "No key found: add TYPESAFE_API_KEY to .env")

key loaded


## One function

Jev has one endpoint. You send `state` (the text) and `questions` (what you want decided).

In [5]:
import json
import urllib.request


def ask(state, questions):
    body = {"model": "jev-latest", "state": state, "questions": questions}
    req = urllib.request.Request(
        "https://api.typesafe.ai/v1/systemone",
        data=json.dumps(body).encode(),
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.load(r)["answers"]

## Example: route a support ticket

A `choice` question picks one option from a list you define.

In [6]:
ticket = "I was charged twice for my subscription this month."

if API_KEY:
    answers = ask(ticket, {
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this ticket?",
            "criteria": {
                "billing": "payments, charges, refunds",
                "technical": "bugs and errors",
                "sales": "pricing and plans",
            },
        },
    })
    print("team:", answers["team"]["choice"])
    print("probabilities:", answers["team"]["probabilities"])
    print("confidence:", answers["team"]["confidence"])

team: billing
probabilities: {'technical': 0.0, 'sales': 0.0, 'billing': 1.0}
confidence: 1.0


## Example: a yes/no question

A `noul` question returns the probability that the statement is true (0 = no, 1 = yes).

In [7]:
if API_KEY:
    answers = ask(ticket, {
        "angry": {"type": "noul", "instructions": "The customer is angry."},
    })
    print("probability the customer is angry:", answers["angry"]["noul"])

probability the customer is angry: 0.55


That's it. Your code decides what to do with the numbers, for example: route only if `confidence > 0.6`, otherwise ask a human.

The full lesson (cascades, guards, LangGraph, calibration) is `notebooks/06_production/48_decision_models_jev.ipynb`.